# 02 — Realtime providers

OpenAI Realtime, Gemini Live, Ultravox, ElevenLabs ConvAI.


## Prerequisites

| Tier | Cells | Required env |
|------|-------|--------------|
| T1+T2 (§1) | always | _none_ |
| T3 (§2) | per-cell | provider keys auto-detected |
| T4 (§3) | gated | `ENABLE_LIVE_CALLS=1` + carrier creds |


In [ ]:
%load_ext autoreload
%autoreload 2

import _setup
env = _setup.load()
print(f'getpatter version target: {env.patter_version}')


## §1: Quickstart

Runs end-to-end with zero API keys.


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline (no network, no carrier calls).


In [ ]:
import sys
import getpatter
with _setup.cell('version_check', tier=1, env=env) as ok:
    if ok:
        print(f'getpatter {getpatter.__version__} on Python {sys.version.split()[0]}')
        assert getpatter.__version__ >= env.patter_version, \
            f'installed {getpatter.__version__} < target {env.patter_version}'


### Local mode
Construct a Patter instance with a Twilio carrier. No API key — runs entirely on your machine.


In [ ]:
from getpatter import Patter, Twilio
with _setup.cell('local_mode', tier=1, env=env) as ok:
    if ok:
        p = Patter(
            carrier=Twilio(
                account_sid='ACtest00000000000000000000000000',
                auth_token='test',
            ),
            phone_number='+15555550100',
            webhook_url='https://example.com/webhook',
        )
        # Verify carrier is wired up via LocalConfig.
        assert p._local_config.telephony_provider == 'twilio'
        assert p._local_config.phone_number == '+15555550100'
        print(f'provider={p._local_config.telephony_provider}  phone={p._local_config.phone_number}')


### Cloud mode (coming soon)
When `api_key=` is supported, Patter cloud handles telephony. For now, the SDK raises `NotImplementedError` — this cell verifies the guard.


In [ ]:
from getpatter import Patter
with _setup.cell('cloud_mode', tier=1, env=env) as ok:
    if ok:
        try:
            Patter(api_key='pt_test_xxx')
            raise AssertionError('expected NotImplementedError — cloud mode guard missing')
        except NotImplementedError as exc:
            print(f'cloud mode guard OK: {exc}')


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline* (STT + LLM + TTS). The factory derives the engine from `engine=` / `stt=`/`tts=`.


In [ ]:
from getpatter import Patter, Twilio, OpenAIRealtime, ElevenLabsConvAI
with _setup.cell('agent_engines', tier=1, env=env) as ok:
    if ok:
        p = Patter(
            carrier=Twilio(account_sid='ACtest00000000000000000000000000',
                           auth_token='test'),
            phone_number='+15555550100',
            webhook_url='https://example.com/webhook',
        )
        rt = p.agent(system_prompt='hi', engine=OpenAIRealtime(api_key='sk-test'))
        cv = p.agent(system_prompt='hi', engine=ElevenLabsConvAI(api_key='el-test', agent_id='a1'))
        pl = p.agent(system_prompt='hi')  # default: OpenAI Realtime fallback
        assert rt.provider == 'openai_realtime', rt.provider
        assert cv.provider == 'elevenlabs_convai', cv.provider
        assert pl.provider == 'openai_realtime', pl.provider
        print(f'rt.provider={rt.provider}  cv.provider={cv.provider}  pl.provider={pl.provider}')


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline (no network, no carrier calls).


### Local mode
Construct a Patter instance with a Twilio carrier. No API key — runs entirely on your machine.


### Cloud mode
Same SDK, just an `api_key=` instead of a carrier — Patter cloud handles telephony.


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline* (STT + LLM + TTS). The factory derives the mode from `engine=` / `stt=`/`tts=`.


## §2: Feature Tour

One cell per feature/provider. Missing keys auto-skip.


## §2 — Feature Tour

Exercises OpenAI Realtime configuration and related SDK primitives.


### OpenAI Realtime agent: full config


In [ ]:
from getpatter import Patter, Twilio, OpenAIRealtime
with _setup.cell('realtime_agent_config', tier=1, env=env) as ok:
    if ok:
        p = Patter(
            carrier=Twilio(account_sid='ACtest00000000000000000000000000', auth_token='test'),
            phone_number='+15555550100',
            webhook_url='https://example.com/webhook',
        )
        agent = p.agent(
            system_prompt='You are a concise assistant.',
            engine=OpenAIRealtime(
                api_key='sk-test',
                model='gpt-4o-realtime-preview',
            ),
            voice='alloy',
            first_message='Hello! How can I help you today?',
            barge_in_threshold_ms=500,
        )
        print(f'provider:              {agent.provider}')
        print(f'voice:                 {agent.voice}')
        print(f'first_message:         {agent.first_message}')
        print(f'barge_in_threshold_ms: {agent.barge_in_threshold_ms}')
        assert agent.provider == 'openai_realtime'
        assert agent.voice == 'alloy'


### SentenceChunker


In [ ]:
from getpatter import SentenceChunker
with _setup.cell('realtime_chunker', tier=1, env=env) as ok:
    if ok:
        sc = SentenceChunker()
        full_response = (
            'The weather today is sunny. Temperature is 72°F. '
            'Humidity is low. Great day for a walk!'
        )
        chunks: list[str] = []
        for char in full_response:
            result = sc.push(char)
            chunks.extend(result)
        chunks.extend(sc.flush())
        print(f'sentences: {len(chunks)}')
        for i, chunk in enumerate(chunks):
            print(f'  [{i}] {chunk.strip()!r}')
        assert len(chunks) >= 2


### Live: OpenAI Realtime models  *(T3 — requires `OPENAI_API_KEY`)*

Connects to OpenAI Realtime WebSocket and lists supported models.


In [ ]:
import httpx
with _setup.cell('openai_realtime_live', tier=3, required=['openai_key'], env=env) as ok:
    if ok:
        resp = httpx.get(
            'https://api.openai.com/v1/models',
            headers={'Authorization': f'Bearer {env.openai_key}'},
            timeout=10,
        )
        resp.raise_for_status()
        models = [m['id'] for m in resp.json()['data'] if 'realtime' in m['id']]
        print(f'OpenAI realtime models: {models[:5]}')
        assert len(models) > 0, 'no realtime models found'


## §3: Live Appendix

Real PSTN calls. Off by default — set `ENABLE_LIVE_CALLS=1`.


## §3 — Live Appendix

Places a real call through OpenAI Realtime. Requires `ENABLE_LIVE_CALLS=1`.


### Pre-flight checklist


In [ ]:
with _setup.cell('live_preflight', tier=4, required=['TWILIO_ACCOUNT_SID', 'TWILIO_AUTH_TOKEN', 'TWILIO_PHONE_NUMBER', 'TARGET_PHONE_NUMBER'], env=env) as ok:
    if ok:
        print(f'  carrier:  Twilio {env.twilio_number}  →  {env.target_number}')
        print(f'  webhook:  {env.public_webhook_url or "(ngrok auto-launch)"}')
        print(f'  engine:   OpenAI Realtime  (gpt-4o-realtime-preview)')


### Live OpenAI Realtime call *(T4)*


In [ ]:
from getpatter import Patter, Twilio, OpenAIRealtime
with _setup.cell('live_realtime_call', tier=4, required=['TWILIO_ACCOUNT_SID', 'TWILIO_AUTH_TOKEN', 'TWILIO_PHONE_NUMBER', 'TARGET_PHONE_NUMBER', 'OPENAI_API_KEY'], env=env) as ok:
    if ok:
        p = Patter(
            carrier=Twilio(account_sid=env.twilio_sid, auth_token=env.twilio_token),
            phone_number=env.twilio_number,
            webhook_url=env.public_webhook_url,
        )
        agent = p.agent(
            system_prompt='You are a demo assistant. Greet the caller and immediately say goodbye.',
            engine=OpenAIRealtime(api_key=env.openai_key, model='gpt-4o-realtime-preview'),
        )
        try:
            await p.call(
                env.target_number,
                agent=agent,
                first_message='Hello! This is a Patter demo call. Goodbye!',
                ring_timeout=env.max_call_seconds,
            )
            print('✓ Realtime call completed')
        finally:
            _setup.hangup_leftover_calls(env)
